# Stage 1–2–3 Scenario Coverage Suite

Stage 4'e geçmeden önce mevcut GPU portunu sekiz farklı propagation case üzerinde test eder:

- Indoor-Office LOS
- Indoor-Office NLOS
- UMi LOS
- UMi NLOS
- UMa LOS
- UMa NLOS
- RMa LOS
- RMa NLOS

Her case için **iki hop da aynı scenario-condition** seçilir:

\[
scenario_{BR}=scenario_{RU}.
\]

Böylece bir test case'i tek bir propagation environment'ını izole eder.

Kontroller:

\[
\text{Stage 1: } \mu_H,\sigma_H^2,d_T,d_R
\]

\[
\text{Stage 2: } \rho_{RB},\rho_{BR},\rho_{RU},\rho_{UR},\rho_{RUhop}
\]

\[
\text{Stage 3: } UBR,\mu_{Feff},\sigma^2_{Feff},C,\mu_{SNR},\sigma^2_{Wick}.
\]

Double parity eşiği:

\[
10^{-10}
\]

Production float32 sanity eşiği:

\[
10^{-4}
\]

(relative error).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, zipfile, json
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

# Modules may be in Drive or uploaded to /content.
module_names = [
    'ris_gpu_physics_stage1.py',
    'ris_gpu_rho_stage2.py',
    'ris_gpu_stats_stage3.py',
]

for name in module_names:
    drive_file = ROOT / name
    content_file = Path('/content') / name

    if drive_file.exists():
        module_dir = drive_file.parent
    elif content_file.exists():
        module_dir = content_file.parent
    else:
        raise FileNotFoundError(
            f"{name} bulunamadı. Drive RIS root'a veya /content altına yükle."
        )

    if str(module_dir) not in sys.path:
        sys.path.insert(0,str(module_dir))

from ris_gpu_physics_stage1 import compare_with_matlab_mat
from ris_gpu_rho_stage2 import compare_rho_matlab_case
from ris_gpu_stats_stage3 import compare_stage3_matlab_case

print("Modules loaded.")

## MATLAB çıktısını yükle

MATLAB'da:

```matlab
run_stage123_scenario_suite
```

çalıştır.

Bu:

```text
stage123_scenario_golden.zip
```

üretecek.

ZIP dosyasını Colab `/content` altına yükle.

In [ ]:
ZIP = Path('/content/stage123_scenario_golden.zip')
assert ZIP.exists(), (
    "stage123_scenario_golden.zip dosyasını /content altına yükle."
)

EXTRACT_ROOT = Path('/content/stage123_suite_extract')
if EXTRACT_ROOT.exists():
    import shutil
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)

with zipfile.ZipFile(ZIP,'r') as zf:
    zf.extractall(EXTRACT_ROOT)

# MATLAB zip() may preserve top-level directory.
candidates = list(EXTRACT_ROOT.rglob('manifest.csv'))
assert len(candidates) == 1, (
    f"manifest.csv tekil bulunamadı: {candidates}"
)

SUITE_ROOT = candidates[0].parent
manifest = pd.read_csv(candidates[0])

display(manifest)

expected = {
    ('Indoor-Office','LOS'),
    ('Indoor-Office','NLOS'),
    ('UMi','LOS'),
    ('UMi','NLOS'),
    ('UMa','LOS'),
    ('UMa','NLOS'),
    ('RMa','LOS'),
    ('RMa','NLOS'),
}

found = set(
    zip(
        manifest['scenario'].astype(str),
        manifest['condition'].astype(str)
    )
)

assert found == expected, (
    f"8-case coverage eksik/fazla.\nExpected={expected}\nFound={found}"
)
assert len(manifest) == 8

print("PASS: exact 8-case scenario coverage")

In [ ]:
# Helpers

def relative_metrics_only(d):
    return {
        k: float(v)
        for k,v in d.items()
        if k.endswith('_relFro') or k.endswith('_rel')
    }

def worst_relative(d):
    vals = list(relative_metrics_only(d).values())
    return max(vals) if vals else np.nan

DOUBLE_LIMIT = 1e-10
FLOAT32_LIMIT = 1e-4

rows = []

device = 'cuda' if torch.cuda.is_available() else 'cpu'

for _,meta in manifest.iterrows():

    case_name = str(meta['caseName'])
    case_dir = SUITE_ROOT / case_name

    required = [
        case_dir/'stage1_br.mat',
        case_dir/'stage1_ru.mat',
        case_dir/'stage2.mat',
        case_dir/'stage3.mat',
    ]
    for p in required:
        assert p.exists(), f"Eksik golden file: {p}"

    # -------- double parity --------
    s1br64 = compare_with_matlab_mat(
        str(case_dir/'stage1_br.mat'),
        device=device,
        parity=True,
    )
    s1ru64 = compare_with_matlab_mat(
        str(case_dir/'stage1_ru.mat'),
        device=device,
        parity=True,
    )
    s264 = compare_rho_matlab_case(
        str(case_dir/'stage2.mat'),
        device=device,
        parity=True,
    )
    s364 = compare_stage3_matlab_case(
        str(case_dir/'stage3.mat'),
        device=device,
        parity=True,
    )

    # -------- production float32 --------
    s1br32 = compare_with_matlab_mat(
        str(case_dir/'stage1_br.mat'),
        device=device,
        parity=False,
    )
    s1ru32 = compare_with_matlab_mat(
        str(case_dir/'stage1_ru.mat'),
        device=device,
        parity=False,
    )
    s232 = compare_rho_matlab_case(
        str(case_dir/'stage2.mat'),
        device=device,
        parity=False,
    )
    s332 = compare_stage3_matlab_case(
        str(case_dir/'stage3.mat'),
        device=device,
        parity=False,
    )

    row = {
        'caseName': case_name,
        'scenario': str(meta['scenario']),
        'condition': str(meta['condition']),
        'fc_GHz': float(meta['fc'])/1e9,
        'nT': int(meta['nT']),
        'nR': int(meta['nR']),
        'nRIS': int(meta['nRIS']),

        'S1_BR_double_worst': worst_relative(s1br64),
        'S1_RU_double_worst': worst_relative(s1ru64),
        'S2_double_worst': worst_relative(s264),
        'S3_double_worst': worst_relative(s364),

        'S1_BR_float32_worst': worst_relative(s1br32),
        'S1_RU_float32_worst': worst_relative(s1ru32),
        'S2_float32_worst': worst_relative(s232),
        'S3_float32_worst': worst_relative(s332),
    }

    row['double_worst_all'] = max(
        row['S1_BR_double_worst'],
        row['S1_RU_double_worst'],
        row['S2_double_worst'],
        row['S3_double_worst'],
    )

    row['float32_worst_all'] = max(
        row['S1_BR_float32_worst'],
        row['S1_RU_float32_worst'],
        row['S2_float32_worst'],
        row['S3_float32_worst'],
    )

    row['double_PASS'] = row['double_worst_all'] < DOUBLE_LIMIT
    row['float32_PASS'] = row['float32_worst_all'] < FLOAT32_LIMIT

    rows.append(row)

summary = pd.DataFrame(rows)

display(
    summary[
        [
            'caseName','fc_GHz','nT','nR','nRIS',
            'S1_BR_double_worst','S1_RU_double_worst',
            'S2_double_worst','S3_double_worst',
            'double_worst_all','double_PASS',
            'float32_worst_all','float32_PASS'
        ]
    ]
)

In [ ]:
# Hard pass / fail

assert summary['double_PASS'].all(), (
    "En az bir scenario case double MATLAB parity testini geçemedi."
)

assert summary['float32_PASS'].all(), (
    "En az bir scenario case float32 production sanity limitini geçemedi."
)

print("================================================")
print(" ALL 8 SCENARIOS: STAGE 1 + 2 + 3 PASS")
print("================================================")
print(
    "Worst double relative error :",
    f"{summary['double_worst_all'].max():.3e}"
)
print(
    "Worst float32 relative error:",
    f"{summary['float32_worst_all'].max():.3e}"
)

print("\nWorst double case:")
display(
    summary.loc[
        [summary['double_worst_all'].idxmax()]
    ]
)

print("\nWorst float32 case:")
display(
    summary.loc[
        [summary['float32_worst_all'].idxmax()]
    ]
)

In [ ]:
# Save compact result for repository / future README.

OUT = Path('/content/stage123_scenario_parity_summary.csv')
summary.to_csv(OUT,index=False)

print("Saved:", OUT)

## Kabul kriteri

Stage 4'e yalnızca şu durumda geçiyoruz:

```text
ALL 8 SCENARIOS: STAGE 1 + 2 + 3 PASS
```

ve

\[
\max e_{\rm double}<10^{-10}.
\]

Float32 için şimdilik daha gevşek bir production sanity guard kullanıyoruz:

\[
\max e_{\rm fp32}<10^{-4}.
\]

Environment tamamen port edildiğinde bu toleransı yalnızca ara bloklarda değil,
nihai `muSNR`, `sigma2Wick`, XGB feature'ları ve q05 üzerinde tekrar değerlendireceğiz.